In [55]:
import pandas as pd
import numpy as np

In [56]:
df = pd.read_csv('../0.dataset/dataset_penjualan_emas_kotor.csv')
df.head(5)

,Transaction_ID,Date,Customer_ID,Gold_Type,Karat,Weight_Gram,Price_Per_Gram,Total_Price,Payment_Method,Store_Location
0,TXN-0295,2024-04-21,CUST-480,Perhiasan,24,35.55,1370210,48710965,Kartu Kredit,JKT Pusat
1,TXN-0809,2024-03-09,CUST-373,Perhiasan,18,NaN,961081,11215815,Transfer Bank,Makassar
2,TXN-0364,2024-11-18,CUST-175,Perhiasan,24,49.59,1323332,65624033,QRIS,Bandung
3,TXN-0869,2025-09-30,CUST-313,Perhiasan,24,44.73,1352769,60509357,QRIS,Makassar
4,TXN-0326,2025-03-13,CUST-158,Koin,18,10.93,960581,10499150,Transfer Bank,Surabaya


In [57]:
df.duplicated().sum()

np.int64(20)

## 1.Mengidentifikasi Duplikat

### 1.Berdasarkan Semua Fitur

In [58]:
duplicated_rows = df.loc[df.duplicated()]

print(f'Baris Duplikat:\n {duplicated_rows.head(5)}')

Baris Duplikat:
     Transaction_ID        Date Customer_ID  Gold_Type  Karat  Weight_Gram  \
122       TXN-0326  2025-03-13    CUST-158       Koin     18        10.93   
186       TXN-0987  2024-09-15    CUST-481   Batangan     18        10.37   
386       TXN-0011  2024-08-02    CUST-335   Batangan     18        41.38   
391       TXN-0489  2024-11-20    CUST-314       Koin     22        31.15   
403       TXN-0903  2024-05-24    CUST-404  Perhiasan     22        43.70   

     Price_Per_Gram  Total_Price Payment_Method Store_Location  
122          960581     10499150  Transfer Bank       Surabaya  
186          979838     10160920   Kartu Kredit  Jakarta Pusat  
386          961439     39784345           QRIS          Medan  
391         1151898     35881622            NaN  jakarta pusat  
403         1169699     51115846  Transfer Bank            SBY  


### 2.Berdasarkan Subset Fitur

In [59]:
duplicated_rows = df.loc[df.duplicated(subset=['Transaction_ID','Customer_ID'])]

print(f"Baris Duplikat berdasarkan 'Transaction_ID','Customer_ID' :\n {duplicated_rows.head(5)}")

Baris Duplikat berdasarkan 'Transaction_ID','Customer_ID' :
     Transaction_ID        Date Customer_ID  Gold_Type  Karat  Weight_Gram  \
122       TXN-0326  2025-03-13    CUST-158       Koin     18        10.93   
186       TXN-0987  2024-09-15    CUST-481   Batangan     18        10.37   
386       TXN-0011  2024-08-02    CUST-335   Batangan     18        41.38   
391       TXN-0489  2024-11-20    CUST-314       Koin     22        31.15   
403       TXN-0903  2024-05-24    CUST-404  Perhiasan     22        43.70   

     Price_Per_Gram  Total_Price Payment_Method Store_Location  
122          960581     10499150  Transfer Bank       Surabaya  
186          979838     10160920   Kartu Kredit  Jakarta Pusat  
386          961439     39784345           QRIS          Medan  
391         1151898     35881622            NaN  jakarta pusat  
403         1169699     51115846  Transfer Bank            SBY  


## 2.Menghapus Duplikat

### 1.Menghapus Baris Duplikat Pertama

In [60]:
df_target = df.copy()
df_target = df_target.drop_duplicates(keep='first')

print(f'Jumlah baris sebelum: {df.shape[0]}')
print(f'Jumlah baris setelah: {df_target.shape[0]}')

Jumlah baris sebelum: 1020
Jumlah baris setelah: 1000


### 2.Menghapus Baris Duplikat Terakhir

In [61]:
df_target = df.copy()
df_target = df_target.drop_duplicates(keep='last')

print(f'Jumlah baris sebelum: {df.shape[0]}')
print(f'Jumlah baris setelah: {df_target.shape[0]}')

Jumlah baris sebelum: 1020
Jumlah baris setelah: 1000


### 3.Menghapus Semua Baris Duplikat

In [62]:
df_target = df.copy()
df_target = df_target.drop_duplicates(keep=False)

print(f'Jumlah baris sebelum: {df.shape[0]}')
print(f'Jumlah baris setelah: {df_target.shape[0]}')

Jumlah baris sebelum: 1020
Jumlah baris setelah: 980


### 4.Menggabungkan Informasi dari Duplikat

In [63]:
df_combined = df_target.groupby(['Transaction_ID', 'Customer_ID']).agg({
    'Weight_Gram': 'sum',          # Menjumlahkan total berat emas dalam transaksi tersebut
    'Total_Price': 'sum',          # Menjumlahkan total uang yang dibayar
    'Date': 'max',                 # Mengambil tanggal transaksi terbaru
    'Store_Location': 'first',     # Lokasi toko pasti sama dalam satu ID transaksi
    'Payment_Method': 'first'      # Metode pembayaran pasti sama
}).reset_index()
df_combined

,Transaction_ID,Customer_ID,Weight_Gram,Total_Price,Date,Store_Location,Payment_Method
0,TXN-0001,CUST-428,22.47,29946218,2024-04-12,Medan,Transfer Bank
1,TXN-0002,CUST-372,22.81,31423238,2025-03-11,Bandung,QRIS
2,TXN-0003,CUST-364,47.95,65303728,2024-09-27,Jakarta Pusat,Tunai
3,TXN-0004,CUST-498,21.84,25275257,2024-04-16,Bandung,Tunai
4,TXN-0005,CUST-137,27.69,31194418,2024-03-12,Jakarta Pusat,None
...,...,...,...,...,...,...,...
975,TXN-0996,CUST-453,12.75,17079543,2025-10-03,Bandung,Transfer Bank
976,TXN-0997,CUST-327,0.00,59534234,2025-08-18,Jakarta Pusat,Transfer Bank
977,TXN-0998,CUST-165,8.65,9723776,2024-11-16,Jakarta Pusat,Kartu Kredit
978,TXN-0999,CUST-434,21.05,23928103,2024-11-14,Bandung,QRIS


## 3.Menangani Duplikat Secara Selektif

### 1.Menghapus Duplikat Berdasarkan Kondisi Spesifik

In [73]:
df_target = df.copy()

#memperbaiki format date
df_target['Date'] = df_target['Date'] = df['Date'].replace("00-00-2024", "2024-01-01")
df_target['Date'] = df_target['Date'] = pd.to_datetime(df_target['Date'])

#mengahapus berdasarkan kondisi baris di kolom 'Weight_Gram','Total_Price' yang terduplikat
df_target = df_target.sort_values('Date').drop_duplicates(subset=['Weight_Gram','Total_Price'],keep='last')

print(f'Jumlah baris sebelum: {df.shape[0]}')
print(f'Jumlah baris setelah: {df_target.shape[0]}')

Jumlah baris sebelum: 1020
Jumlah baris setelah: 1000


### 2.Menandai Duplikat untuk Analisis Lanjutan

In [72]:
df_target = df.copy()
df_target['isDuplicated'] = df.duplicated(keep=False)

print(f'Data yang terduplikat:\n{df_target.loc[df_target["isDuplicated"]].head(5)}')

Data yang terduplikat:
   Transaction_ID        Date Customer_ID Gold_Type  Karat  Weight_Gram  \
4        TXN-0326  2025-03-13    CUST-158      Koin     18        10.93   
15       TXN-0144  2024-08-28    CUST-191      Koin     22        44.29   
27       TXN-0489  2024-11-20    CUST-314      Koin     22        31.15   
33       TXN-0987  2024-09-15    CUST-481  Batangan     18        10.37   
43       TXN-0011  2024-08-02    CUST-335  Batangan     18        41.38   

    Price_Per_Gram  Total_Price Payment_Method Store_Location  isDuplicated  
4           960581     10499150  Transfer Bank       Surabaya          True  
15         1159884     51371262   Kartu Kredit       Surabaya          True  
27         1151898     35881622            NaN  jakarta pusat          True  
33          979838     10160920   Kartu Kredit  Jakarta Pusat          True  
43          961439     39784345           QRIS          Medan          True  
